# Machine Learning Practical Implementation
## Week 7 & 8: Regression & Classification Models

### Objective:
1. **Regression Models**:
   - Linear Regression
   - Polynomial Regression (Degree 2)
   - Ridge Regression (L2 Regularization)
   - Lasso Regression (L1 Regularization)
2. **Classification Models**:
   - Logistic Regression
   - K-Nearest Neighbors (KNN)

**Dataset**: Employee Salary Dataset (employee_salary_dataset.csv)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Load and Explore Dataset

In [ ]:
df = pd.read_csv('employee_salary_dataset.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()
df.describe()


## 2. Regression Models Setup
- **Target Variable ($)**: Monthly_Salary (Continuous)
- **Features ($)**: Experience_Years, Age, Department, Education_Level, Gender, City


In [ ]:
features_df = df.drop(columns=['EmployeeID', 'Name', 'Monthly_Salary'])
target_reg = df['Monthly_Salary']

categorical_cols = ['Department', 'Education_Level', 'Gender', 'City']
numerical_cols = ['Experience_Years', 'Age']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    features_df, target_reg, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)


### 2.1 Linear Regression

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
lr_pipeline.fit(X_train_reg, y_train_reg)
y_pred_lr = lr_pipeline.predict(X_test_reg)

print('--- Linear Regression Results ---')
print(f'Train R2: {r2_score(y_train_reg, lr_pipeline.predict(X_train_reg)):.4f}')
print(f'Test R2:  {r2_score(y_test_reg, y_pred_lr):.4f}')
print(f'MAE:      {mean_absolute_error(y_test_reg, y_pred_lr):.2f}')
print(f'RMSE:     {np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)):.2f}')


### 2.2 Polynomial Regression (Degree 2)

In [ ]:
preprocessor_poly = ColumnTransformer(
    transformers=[
        ('num_poly', Pipeline([
            ('scaler', StandardScaler()),
            ('poly', PolynomialFeatures(degree=2, include_bias=False))
        ]), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)

poly_pipeline = Pipeline([
    ('preprocessor', preprocessor_poly),
    ('regressor', LinearRegression())
])
poly_pipeline.fit(X_train_reg, y_train_reg)
y_pred_poly = poly_pipeline.predict(X_test_reg)

print('--- Polynomial Regression Results ---')
print(f'Train R2: {r2_score(y_train_reg, poly_pipeline.predict(X_train_reg)):.4f}')
print(f'Test R2:  {r2_score(y_test_reg, y_pred_poly):.4f}')
print(f'MAE:      {mean_absolute_error(y_test_reg, y_pred_poly):.2f}')
print(f'RMSE:     {np.sqrt(mean_squared_error(y_test_reg, y_pred_poly)):.2f}')


### 2.3 Ridge Regression (L2 Regularization)

In [ ]:
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])
ridge_pipeline.fit(X_train_reg, y_train_reg)
y_pred_ridge = ridge_pipeline.predict(X_test_reg)

print('--- Ridge Regression Results ---')
print(f'Train R2: {r2_score(y_train_reg, ridge_pipeline.predict(X_train_reg)):.4f}')
print(f'Test R2:  {r2_score(y_test_reg, y_pred_ridge):.4f}')
print(f'MAE:      {mean_absolute_error(y_test_reg, y_pred_ridge):.2f}')
print(f'RMSE:     {np.sqrt(mean_squared_error(y_test_reg, y_pred_ridge)):.2f}')


### 2.4 Lasso Regression (L1 Regularization)

In [ ]:
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=500.0, max_iter=10000, random_state=42))
])
lasso_pipeline.fit(X_train_reg, y_train_reg)
y_pred_lasso = lasso_pipeline.predict(X_test_reg)

print('--- Lasso Regression Results ---')
print(f'Train R2: {r2_score(y_train_reg, lasso_pipeline.predict(X_train_reg)):.4f}')
print(f'Test R2:  {r2_score(y_test_reg, y_pred_lasso):.4f}')
print(f'MAE:      {mean_absolute_error(y_test_reg, y_pred_lasso):.2f}')
print(f'RMSE:     {np.sqrt(mean_squared_error(y_test_reg, y_pred_lasso)):.2f}')


## 3. Classification Models Setup
- **Target Variable ($)**: High_Salary (1 if Monthly_Salary > Median [73890.50], 0 otherwise)
- **Features ($)**: Experience_Years, Age, Department, Education_Level, Gender, City


In [ ]:
salary_median = df['Monthly_Salary'].median()
df['High_Salary'] = (df['Monthly_Salary'] > salary_median).astype(int)
target_clf = df['High_Salary']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    features_df, target_clf, test_size=0.2, random_state=42, stratify=target_clf
)


### 3.1 Logistic Regression

In [ ]:
log_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])
log_reg.fit(X_train_clf, y_train_clf)
y_pred_lr_clf = log_reg.predict(X_test_clf)

print('--- Logistic Regression Results ---')
print(f'Accuracy:  {accuracy_score(y_test_clf, y_pred_lr_clf):.4f}')
print(f'Precision: {precision_score(y_test_clf, y_pred_lr_clf):.4f}')
print(f'Recall:    {recall_score(y_test_clf, y_pred_lr_clf):.4f}')
print(f'F1-Score:  {f1_score(y_test_clf, y_pred_lr_clf):.4f}')
print('
Classification Report:
', classification_report(y_test_clf, y_pred_lr_clf))


### 3.2 K-Nearest Neighbors (KNN)

In [ ]:
knn = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])
knn.fit(X_train_clf, y_train_clf)
y_pred_knn = knn.predict(X_test_clf)

print('--- KNN (k=5) Results ---')
print(f'Accuracy:  {accuracy_score(y_test_clf, y_pred_knn):.4f}')
print(f'Precision: {precision_score(y_test_clf, y_pred_knn):.4f}')
print(f'Recall:    {recall_score(y_test_clf, y_pred_knn):.4f}')
print(f'F1-Score:  {f1_score(y_test_clf, y_pred_knn):.4f}')
print('
Classification Report:
', classification_report(y_test_clf, y_pred_knn))
